# Scenario - 3


In [13]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


In [14]:

import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
from dotenv import load_dotenv
from huggingface_hub import login
 
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)


MODEL_ID = "microsoft/phi-4"
# MODEL_ID = "Qwen/Qwen3-14B"
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
# MODEL_ID = "mistralai--Mistral-7B-v0.1"
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated -- must accept license + login() above
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights:   1%|          | 2/243 [00:00<00:22, 10.71it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 243/243 [00:02<00:00, 87.74it/s]


Model loaded.


# Load files

In [15]:
import os
import json
import re
import torch
import pandas as pd

  
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"


EDGE_LIST_NAMES = "../edge_list_top10pos_names.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}

DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768
MAX_NEW_TOKENS = 800

In [16]:
# with open(STRUCTURE_PROMPTS_JSON) as f:
with open(EDGE_LIST_NAMES) as f:
# with open(EDGE_LIST_NUMBERS) as f:
    structure_prompts_all = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")


subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")
print(brain_graph_texts[subject_ids[0]])

Loaded 116 AAL116 ROI names
Loaded 768 subjects, running on first 768
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
Node feature:
Frontal_Med_Orb_R: 0.323
Temporal_Sup_R: 0.298
Temporal_Sup_L: 0.296
Insula_L: 0.294
Occipital_Inf_L: 0.293
Parietal_Sup_L: 0.289
Cingulum_Post_L: 0.286
Frontal_Med_Orb_L: 0.284
Cingulum_Post_R: 0.283
Fusiform_R: 0.280
Cerebelum_Crus1_R: 0.278
Temporal_Pole_Mid_R: 0.277
SupraMarginal_L: 0.276
Cerebelum_Crus1_L: 0.275
Frontal_Sup_Medial_R: 0.273
Frontal_Sup_Orb_L: 0.273
Frontal_Inf_Oper_L: 0.268
Heschl_L: 0.268
Putamen_L: 0.268
Frontal_Sup_Medial_L: 0.265
Temporal_Pole_Mid_L: 0.264
Occipital_Mid_L: 0.264
Frontal_Sup_L: 0.264
Occipital_Mid_R: 0.258
Parietal_Sup_R: 0.258
Rolandic_Oper_L: 0.255
Angular_L: 0.255
Paracentral_Lobule_R: 0.254
Temporal_Pole_Sup_R: 0.252
Occipital_Inf_R: 0.248
Fusiform_L: 0.243
Postcentral_R: 0.239
Frontal_Inf_Tri_L: 0.238
Temporal_Pole_Sup_L: 0.238
Frontal_Inf_Orb_L: 0.238
Insula_R: 0.238
Postcentral

## 3. Prompt Template -- Description / Task / Request

In [17]:
def build_stage1_prompt(subject_id, brain_graph_text):

    return f"""Template: This data comes from the {DATASET_NAME} dataset, preprocessed using the {PREPROCESS_TEMPLATE} brain atlas template (116 regions of interest).
Description: The following is a functional connectivity brain graph for one subject, extracted from resting-state fMRI. Nodes represent brain regions, identified by their anatomical region name (AAL atlas); edges represent the strength of functional connectivity between two named regions.
Task: {TASK_DESC}
Request: Analyze the BrainGraph node features and edge list below. Find the main connectivity patterns and the most important features. Then predict whether the subject belongs to Control or ADHD. Give a confidence score (0–1) for both classes. Your output must strictly follow this JSON structure and nothing else:
{{
"analysis": "a brief description of the overall connectivity pattern observed in the data (1-2 sentences)",
"key_features": ["feature 1", "feature 2", "..."],
"prediction": "{LABEL1} or {LABEL2}",
"class_confidence": {{"{LABEL1}": 0.0,"{LABEL2}": 0.0}}
}}
BrainGraph (Text format):
{brain_graph_text}"""

first_sid = subject_ids[0]
first_prompt = build_stage1_prompt(first_sid, brain_graph_texts[first_sid])
print(f"=== INPUT SANITY CHECK -- Subject {first_sid} ===")
print(first_prompt)
print(f"\n--- prompt length: {len(first_prompt)} chars, "
      f"{len(tokenizer(first_prompt)['input_ids'])} tokens ---")
input_tokens = tokenizer.encode(first_prompt)

total_input_tokens = len(input_tokens)

print(f"Total input tokens: {total_input_tokens}")


=== INPUT SANITY CHECK -- Subject 1018959 ===
Template: This data comes from the ADHD-200 dataset, preprocessed using the AAL116 brain atlas template (116 regions of interest).
Description: The following is a functional connectivity brain graph for one subject, extracted from resting-state fMRI. Nodes represent brain regions, identified by their anatomical region name (AAL atlas); edges represent the strength of functional connectivity between two named regions.
Task: interpreting functional connectivity patterns and reasoning about the most likely diagnostic group
Request: Analyze the BrainGraph node features and edge list below. Find the main connectivity patterns and the most important features. Then predict whether the subject belongs to Control or ADHD. Give a confidence score (0–1) for both classes. Your output must strictly follow this JSON structure and nothing else:
{
"analysis": "a brief description of the overall connectivity pattern observed in the data (1-2 sentences)",
"k

## 4. Generate -- one subject per call, human-readable parsed output

In [18]:
def parse_stage1_json(raw_text):
    # grab the reasoning trace, if the model produced one
    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else None

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    if not isinstance(parsed, dict):
        return None

    # put the thinking trace into "analysis" — overwrite whatever the model wrote there
    if reasoning:
        parsed["analysis"] = reasoning

    return parsed

In [19]:
OUTPUT_DIR = "top_pos10/phi"

CHECKPOINT_PATH = "phi_pos10_fc_named_scn3_checkpoint.json"
FINAL_PATH = "phi_pos10_fc_named_scn3_prompts.json"
LOG_PATH = "phi_pos10_fc_named_scn3_prompts_errors.log"

SAVE_EVERY = 10          # write checkpoint to disk after every N *successful* subjects
RETRY_ATTEMPTS = 2       # attempts per subject within a single run, before moving on


In [20]:
# import atexit
# import gc
# import json
# import os
# import signal
# import time

# import torch
# from tqdm.auto import tqdm



# def load_checkpoint():
#     if os.path.exists(CHECKPOINT_PATH):
#         with open(CHECKPOINT_PATH, "r") as f:
#             loaded = json.load(f)
#         # Only trust entries that actually completed successfully -- guards
#         # against an old-format checkpoint that might contain error records.
#         loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
#         print(f"Resuming: {len(loaded)} subjects already completed successfully.")
#         return loaded
#     return {}


# def save_checkpoint(results, path=CHECKPOINT_PATH):
#     tmp_path = path + ".tmp"
#     with open(tmp_path, "w") as f:
#         json.dump(results, f, indent=2)
#     os.replace(tmp_path, path)  # atomic write


# def log_line(msg):
#     ts = time.strftime("%Y-%m-%d %H:%M:%S")
#     with open(LOG_PATH, "a") as f:
#         f.write(f"[{ts}] {msg}\n")


# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)
#     messages = [{"role": "user", "content": prompt}]
#     prompt_text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

#     THINK_BUDGET = 500          # tokens allowed for reasoning
#     ANSWER_BUDGET = max_new_tokens - THINK_BUDGET   # remaining ~300 for the JSON

#     try:
#         with torch.no_grad():
#             # Stage 1: let it think, but cap it
#             think_ids = model.generate(
#                 **inputs, max_new_tokens=THINK_BUDGET,
#                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
#             )
#             think_text = tokenizer.decode(
#                 think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
#             )
#             # force-close thinking if it didn't finish on its own
#             if "</think>" not in think_text:
#                 think_text += "\n</think>\n"

#             # Stage 2: continue from forced-closed think, now answer in JSON
#             stage2_prompt = prompt_text + think_text
#             inputs2 = tokenizer(stage2_prompt, return_tensors="pt").to(model.device)
#             answer_ids = model.generate(
#                 **inputs2, max_new_tokens=ANSWER_BUDGET,
#                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
#             )
#             answer_text = tokenizer.decode(
#                 answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True
#             )
#         return think_text + answer_text
#     finally:
#         del inputs
#         gc.collect()
#         torch.cuda.empty_cache()


# def process_one_subject(sid):
#     """Returns a result dict on success, or raises the last exception after
#     exhausting RETRY_ATTEMPTS."""
#     last_exc = None
#     for attempt in range(1, RETRY_ATTEMPTS + 1):
#         try:
#             raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
#             parsed = parse_stage1_json(raw)

#             gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
#             ground_truth = "Control" if gt == 0 else "ADHD"

#             if parsed is None:
#                 # Model ran fine but didn't return valid JSON -- this is NOT
#                 # a CUDA error, retrying won't help, log it and give up on
#                 # this subject for this run (still excluded from JSON).
#                 log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
#                 raise ValueError("Could not parse JSON from model output")

#             return {
#                 "subject_id": sid,
#                 "ground_truth": ground_truth,
#                 "prediction": parsed.get("prediction", ""),
#                 "class_confidence": parsed.get("class_confidence", {}),
#                 "analysis": parsed.get("analysis", ""),
#                 "key_features": parsed.get("key_features", []),
#                 # "conclusion": parsed.get("conclusion", ""),
#             }

#         except Exception as e:
#             last_exc = e
#             log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
#             gc.collect()
#             torch.cuda.empty_cache()
#             torch.cuda.synchronize()

#     raise last_exc

In [21]:
import gc
import json
import os
import time

import torch
from tqdm.auto import tqdm



def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        # Only trust entries that actually completed successfully -- guards
        # against an old-format checkpoint that might contain error records.
        loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
        print(f"Resuming: {len(loaded)} subjects already completed successfully.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)  # atomic write


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")

def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_stage1_prompt(subject_id, brain_graph_text)
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        answer_text = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return answer_text
    finally:
        del inputs
        gc.collect()
        torch.cuda.empty_cache()



# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS, retries=2):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)  # unchanged prompt
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": prompt}],
#                 max_completion_tokens=max_new_tokens,   # was max_tokens
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))

def process_one_subject(sid):
    """Returns a result dict on success, or raises the last exception after
    exhausting RETRY_ATTEMPTS."""
    last_exc = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_stage1_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed is None:
                # Model ran fine but didn't return valid JSON -- this is NOT
                # a CUDA error, retrying won't help, log it and give up on
                # this subject for this run (still excluded from JSON).
                log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
                raise ValueError("Could not parse JSON from model output")

            return {
                "subject_id": sid,
                "ground_truth": ground_truth,
                "prediction": parsed.get("prediction", ""),
                "class_confidence": parsed.get("class_confidence", {}),
                "analysis": parsed.get("analysis", ""),
                "key_features": parsed.get("key_features", []),
                # "conclusion": parsed.get("conclusion", ""),
            }

        except Exception as e:
            last_exc = e
            log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    raise last_exc


## Snaity Check

In [22]:
# ---------------------------------------------------------------------
# Sanity check — run this before the main inference loop
# ---------------------------------------------------------------------
assert len(subject_ids) == N_SUBJECTS, f"Expected {N_SUBJECTS} subjects, got {len(subject_ids)}"
assert all(sid in brain_graph_texts for sid in subject_ids), "Missing brain_graph_texts for some subject_ids"
assert meta is not None, "meta not loaded"
assert set(subject_ids) <= set(meta["subject_id"]), "Some subject_ids missing from meta"

gt_counts = meta.loc[meta["subject_id"].isin(subject_ids), "DX_binary"].value_counts()
print(f"Ground truth check: Control={gt_counts.get(0,0)}, ADHD={gt_counts.get(1,0)} "
      f"(expected 488 / 280)")



# one live generation, checked against the schema the prompt actually asks for
_test_sid = subject_ids[0]
_raw = generate_stage1_response_safe(_test_sid, brain_graph_texts[_test_sid])
_parsed = parse_stage1_json(_raw)
assert _parsed is not None, f"Sanity-check generation failed to parse JSON:\n{_raw[:500]}"
_expected_keys = {"analysis", "key_features", "prediction", "class_confidence"}
_missing = _expected_keys - _parsed.keys()
assert not _missing, f"Parsed JSON missing expected keys: {_missing}\nGot keys: {list(_parsed.keys())}\nRaw output:\n{_raw[:800]}"
assert _parsed["prediction"] in {"Control", "ADHD"}, f"Unexpected prediction value: {_parsed['prediction']}"

print(f"Sanity check passed on subject {_test_sid} — prediction={_parsed['prediction']}, "
      f"keys OK, {len(subject_ids)} subjects ready, output -> {FINAL_PATH}")

      
sid = subject_ids[0]  # or any specific ID, e.g. "1018959"

raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
print(repr(raw))  # exact raw text, for debugging if parsing fails

parsed = parse_stage1_json(raw)

if parsed is None:
    print("\nPARSE FAILED -- raw text above was not valid JSON.")
else:
    print("\nPARSED SUCCESSFULLY:")
    print(f"  prediction:       {parsed.get('prediction')}")
    print(f"  class_confidence: {parsed.get('class_confidence')}")
    print(f"  key_features:     {parsed.get('key_features')}")
    print(f"  analysis:         {parsed.get('analysis')}")

Ground truth check: Control=488, ADHD=280 (expected 488 / 280)


/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Sanity check passed on subject 1018959 — prediction=Control, keys OK, 768 subjects ready, output -> phi_pos10_fc_named_scn3_prompts.json
'```json\n{\n  "analysis": "The connectivity pattern shows strong connections between homologous regions across hemispheres, with notable connectivity in frontal, temporal, and parietal regions, suggesting typical resting-state network organization.",\n  "key_features": [\n    "Strong homologous connectivity (e.g., Temporal_Sup_L:0.92:Temporal_Sup_R)",\n    "High connectivity in frontal and temporal regions",\n    "Moderate connectivity in parietal and occipital regions"\n  ],\n  "prediction": "Control",\n  "class_confidence": {"Control": 0.85, "ADHD": 0.15}\n}\n```\n\nThis analysis is based on the observed connectivity patterns, which align more closely with typical resting-state networks seen in control subjects, characterized by strong inter-hemispheric connections and robust connectivity in frontal and temporal regions. The confidence score reflec

## Inference

In [23]:
# ---------------------------------------------------------------------
# Resume from checkpoint, figure out what's left to do
# ---------------------------------------------------------------------
results = load_checkpoint()
remaining_ids = [sid for sid in subject_ids if sid not in results]
print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

n_success, n_parse_fail, n_error = 0, 0, 0
start_time = time.time()

with open(LOG_PATH, "a") as log_f:
    pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
    for sid in pbar:
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_stage1_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed:
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "prediction": parsed.get("prediction", ""),
                    "class_confidence": parsed.get("class_confidence", {}),
                    "analysis": parsed.get("analysis", ""),
                    "key_features": parsed.get("key_features", []),
                    # "conclusion": parsed.get("conclusion", ""),
                }
                n_success += 1
            else:
                # Model ran fine, just didn't return valid JSON -- this
                # counts as "done", NOT retried. Raw output is kept so
                # you can inspect/re-parse it later if needed.
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "raw_output": raw,
                    "parsed": None,
                }
                n_parse_fail += 1
                log_f.write(f"[parse_fail] {sid}: {raw[:300]!r}\n")
                log_f.flush()

        except Exception as e:
            # A real crash (CUDA error, etc.) -- NOT saved to results, stays
            # "not done", so this exact subject is retried the next time
            # this cell runs.
            n_error += 1
            log_f.write(f"[error] {sid}: {repr(e)}\n")
            log_f.flush()
            gc.collect()
            torch.cuda.empty_cache()

        done = len(results)
        pbar.set_postfix(done=done, ok=n_success, parse_fail=n_parse_fail, err=n_error)

        if done % SAVE_EVERY == 0:
            save_checkpoint(results)

# Final save + promote checkpoint to the "done" filename once everything ran
save_checkpoint(results)
elapsed = time.time() - start_time
print(f"\nFinished this run in {elapsed/60:.1f} min.")
print(f"Total saved: {len(results)}/{len(subject_ids)}  "
      f"(ok={n_success}, parse_fail={n_parse_fail}, error={n_error})")

if len(results) == len(subject_ids):
    os.replace(CHECKPOINT_PATH, FINAL_PATH)
    print(f"All subjects done -> {FINAL_PATH}")
else:
    print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")


Total subjects: 768 | already done: 0 | remaining: 768


Subjects: 100%|██████████| 768/768 [2:03:10<00:00,  9.62s/subj, done=768, err=0, ok=768, parse_fail=0]  


Finished this run in 123.2 min.
Total saved: 768/768  (ok=768, parse_fail=0, error=0)
All subjects done -> phi_pos10_fc_named_scn3_prompts.json


In [24]:
import json, os
# FINAL_PATH = "bleg_llama_all_subjects_named_prompts_2.json"

VALID_LABELS = {"ADHD", "Control"}


# FINAL_PATH = "bleg_llama_all_subjects_named_prompts_3.json"

def print_stats(data):
    n = len(data)

    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed_entries = [
        (sid, r.get("prediction"))
        for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    n_malformed_pred = len(malformed_entries)

    # scorable = BOTH ground_truth and prediction are exactly "ADHD" or "Control"
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_scorable = len(scorable)

    n_gt_adhd_all = sum(1 for r in data.values() if r.get("ground_truth") == "ADHD")
    n_gt_control_all = sum(1 for r in data.values() if r.get("ground_truth") == "Control")

    n_pred_adhd = sum(1 for r in scorable if r["prediction"] == "ADHD")
    n_pred_control = sum(1 for r in scorable if r["prediction"] == "Control")

    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    # Per-class, denominator restricted to scorable subjects only
    scorable_gt_adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    scorable_gt_control = [r for r in scorable if r["ground_truth"] == "Control"]
    n_adhd_correct = sum(1 for r in scorable_gt_adhd if r["prediction"] == "ADHD")
    n_control_correct = sum(1 for r in scorable_gt_control if r["prediction"] == "Control")

    print(f"Total subjects in file:              {n}")
    print(f"  -- no prediction (parse failure):  {n_no_pred}")
    print(f"  -- malformed prediction (neither ADHD nor Control exactly): {n_malformed_pred}")
    if malformed_entries:
        print(f"     malformed subject_ids and their prediction values:")
        for sid, val in malformed_entries:
            print(f"       {sid}: {val!r}")
    print(f"  -- scorable (valid ground_truth AND valid prediction): {n_scorable}")
    print()
    print(f"Ground truth (all {n} subjects) -- ADHD: {n_gt_adhd_all} | Control: {n_gt_control_all}")
    print(f"Prediction (scorable subjects only, n={n_scorable}) -- ADHD: {n_pred_adhd} | Control: {n_pred_control}")
    print()
    if n_scorable:
        print(f"Accuracy (scorable only): {n_correct}/{n_scorable} = {n_correct / n_scorable:.2%}")
    else:
        print("Accuracy: n/a (no scorable subjects)")

    print()
    print("Correctly predicted, by class (denominator = scorable subjects of that class only):")
    if scorable_gt_adhd:
        print(f"  ADHD    correctly predicted: {n_adhd_correct}/{len(scorable_gt_adhd)} = {n_adhd_correct / len(scorable_gt_adhd):.2%}")
    else:
        print("  ADHD    correctly predicted: n/a")
    if scorable_gt_control:
        print(f"  Control correctly predicted: {n_control_correct}/{len(scorable_gt_control)} = {n_control_correct / len(scorable_gt_control):.2%}")
    else:
        print("  Control correctly predicted: n/a")


if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        data = json.load(f)
    print("RUN COMPLETE.")
    print_stats(data)
elif os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        data = json.load(f)
    print_stats(data)

else:
    print("No output file found yet -- nothing processed.")

print(FINAL_PATH)


RUN COMPLETE.
Total subjects in file:              768
  -- no prediction (parse failure):  0
  -- malformed prediction (neither ADHD nor Control exactly): 0
  -- scorable (valid ground_truth AND valid prediction): 768

Ground truth (all 768 subjects) -- ADHD: 280 | Control: 488
Prediction (scorable subjects only, n=768) -- ADHD: 564 | Control: 204

Accuracy (scorable only): 308/768 = 40.10%

Correctly predicted, by class (denominator = scorable subjects of that class only):
  ADHD    correctly predicted: 192/280 = 68.57%
  Control correctly predicted: 116/488 = 23.77%
phi_pos10_fc_named_scn3_prompts.json
